# Geometric & Drawing Operations (TP 1 — Exercise 4)

## Assignment Objectives
Operate on `tp1_exo4_exo5_peppers_color.bmp`:
1. read `im1` and extract its dimensions `(h, w)`;
2. resize to **half size** (`im2`);
3. crop the region `[30:150, 200:400]` (`im3`);
4. draw a **blue** rectangle with the green caption `"Mon rectangle"` (`im4`);
5. rotate by 90° (`im5`) and apply Gaussian blur (`im6`).

## Mathematical Principles & Intuition
- **Resize (area interpolation):** $I' = I \downarrow (w/2, h/2)$ averages the
  $2\times2$ blocks of the source — the anti-aliased choice for shrinking.
- **Crop:** pure NumPy slicing $I[30:150,\,200:400]$ — a *view* unless copied;
  `.copy()` detaches it so later in-place edits cannot corrupt `im1`.
- **Rotation by 90° (clockwise):** the transpose + vertical flip $I' = I^{\top}$ flipped, i.e. $I'(y,x) = I(W-1-x, y)$.
- **Gaussian blur:** convolution with the isotropic kernel
  $$G_\sigma(x,y) = \frac{1}{2\pi\sigma^2} e^{-(x^2+y^2)/2\sigma^2}$$
  which attenuates high frequencies — each pixel becomes a weighted average of
  its neighbourhood.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Step 1: The Six Derived Images


In [ ]:
# 1) Read im1 and extract (h, w)
im1 = load_image("tp1_exo4_exo5_peppers_color.bmp", cv2.IMREAD_COLOR)
h, w = im1.shape[:2]
print(f"im1 dimensions: h = {h}, w = {w}")

# 2) Resize to half size
im2 = cv2.resize(im1, (w // 2, h // 2), interpolation=cv2.INTER_AREA)

# 3) Crop [y 30:150, x 200:400]
im3 = im1[30:150, 200:400].copy()

# 4) Copy im1, draw a blue rectangle + green caption
im4 = im1.copy()
cv2.rectangle(im4, (200, 30), (400, 150), (255, 0, 0), thickness=2)  # blue in BGR
cv2.putText(im4, "Mon rectangle", (200, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
            (0, 255, 0), 2, cv2.LINE_AA)                              # green in BGR

# 5) Rotate 90 degrees clockwise
im5 = cv2.rotate(im1, cv2.ROTATE_90_CLOCKWISE)

# 6) Gaussian blur
im6 = cv2.GaussianBlur(im1, (7, 7), sigmaX=1.5)

show_images([
    (f"im1 original {im1.shape[:2]}", im1),
    (f"im2 resized /2 {im2.shape[:2]}", im2),
    (f"im3 crop {im3.shape[:2]}", im3),
    ("im4 rectangle + text", im4),
    (f"im5 rotation 90 deg {im5.shape[:2]}", im5),
    ("im6 Gaussian blur", im6),
], cols=3)


In [ ]:
# Quantitative verification table
from TAI.utils.metrics import calculate_psnr

rows = [
    ("im2 = resize(im1, 1/2)", im1, im2),
    ("im3 = crop(im1)", im1, im3),
    ("im5 = rot90(im1)", im1, im5),
    ("im6 = blur(im1)", im1, im6),
]
print(f"{'Operation':24s} | {'Shape':14s} | {'PSNR vs im1 (dB)':>16s}")
print("-" * 62)
for label, ref, out in rows:
    psnr_val = calculate_psnr(ref[: out.shape[0], : out.shape[1]], out) if ref.shape != out.shape else calculate_psnr(ref, out)
    print(f"{label:24s} | {str(out.shape[:2]):14s} | {psnr_val:16.2f}")

mean_shift = float(np.mean(im6.astype(np.float64) - im1.astype(np.float64)))
print(f"\nGaussian blur mean intensity shift: {mean_shift:+.3f} (≈ 0 — blur preserves the DC level)")


## Analysis & Key Takeaways
- Cropping is a zero-cost NumPy **view**; `.copy()` is required to keep the crop independent from later modifications of `im1`.
- `cv2.resize` takes the **(width, height)** target — the transpose of `img.shape`; mixing them silently transposes the image.
- Rotation by 90° is lossless (pure index remapping), while area-resizing and Gaussian blur are low-pass operations: fine textures lose energy but the DC level (mean) is preserved.
- Drawing helpers (`cv2.rectangle`, `cv2.putText`) mutate their argument in place — hence the defensive `im4 = im1.copy()`.
